# Actividad 2

# Información General del Dataset

**Nombre:** *Student Performance Factors Dataset*

**Fuente y liga de acceso:** *https://www.kaggle.com/datasets/mosapabdelghany/student-performance-factors-dataset*

**Fecha de consulta:** *13 de agosto de 2026 — 17 de agosto de 2026*

**Fecha de última actualización en Kaggle:** *Hace 10 meses*

**Descripción:** *Este conjunto contiene variables numéricas y categóricas relacionadas con factores académicos, personales, familiares y esoclares que pueden influir en el desempeño de los estudiantes. En esta actividad se tendrá como objetivo seleccionar las variables más importantes y entrenar los modelos para que puedan ser capaces de predecir cuáles variables son las que verdaderamente influyen.*

# Datos del Dataset

## Explicación de los registros

En este caso se usará el dataset limpio que se obtuvo de la actividad 1. Éstos son los registros originales y nuevos.

**Registros originales**
* **Hours_Studied:** *Número de horas de estudio por semana*
* **Attendance:** *Porcentaje de asistencia en el periodo*
* **Parental_Involvment:** *Nivel de apoyo académico por parte de los padres*
* **Access_to_Resources:** *Disponibilidad de recursos académicos como libros o internet*
* **Extracurricular_Activities:** *Si el estudiante está involucrado en actividades extracurriculares*
* **Sleep_Hours:** *Promedio de horas de sueño al día*
* **Previous_Scores:** *Promedio de calificaciones académicas previas*
* **Motivation_Level:** *Nivel de motivación del estudiante*
* **Internet_Access:** *Si el estudiante tiene acceso al internet*
* **Tutoring_Sessions:** *Número de sesiones de tutoría a la semana*
* **Family_Income:** *Nivel de ingresos familiares*
* **Teacher_Quality:** *Calidad general percibida de los docentes*
* **School_Type:** *Tipo de escuela (Pública o Privada)*
* **Peer_Influence:** *Influencia de los compañeros en el rendimiento académico*
* **Physical_Activity:** *Horas de actividad física a la semana*
* **Learning_Disabilities:** *Si el estudiante presenta dificultades de aprendizaje*
* **Parental_Education_Level:** *Nivel máximo de estudios de los padres*
* **Distance_from_Home:** *Distancia entre el hogar y la escuela (Cerca, Moderada, Lejos)*
* **Gender:** *Género del estudiante*
* **Exam_Score:** *Calificación final del examen*

### Variable objetivo
La variable **Exam_Score** es nuestra variable objetivo.

### Pregunta predictiva que guiará el análisis
¿Qué variables son las fundamentales para que un estudiante pueda obtener un puntaje alto en su examen? ¿Existe una combinación de variables que aumente o disminuya este puntaje?

# Paso 1 y 2

In [22]:
%pip install matplotlib seaborn pandas scikit-learn kagglehub

Note: you may need to restart the kernel to use updated packages.


In [23]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import kagglehub
from kagglehub import KaggleDatasetAdapter

### Limpieza que se realizó en la Actividad 1

In [ ]:
def clean_data():
    # Load the dataset from Kaggle
    file_path = "StudentPerformanceFactors.csv"
    
    df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        "mosapabdelghany/student-performance-factors-dataset",
        file_path,
    )
    
    # Columns of type string with missing values
    df_objects = df.select_dtypes(include=['object'])
    missing_columns = df_objects.columns[df_objects.isnull().any()].tolist()
    
    # Columns of type int64 with missing values
    #df_int64 = df.select_dtypes(include=['int64'])
    #missing_columns += df_int64.columns[df_int64.isnull().any()].tolist()
    
    # Print the number of missing values for each column with missing values
    for column in missing_columns:
        print(f"Valores faltantes en '{column}': {df[column].isnull().sum()}")
    
    # Fill missing values with the mode of the respective columns
    for column in missing_columns:
        df[column] = df[column].fillna(df[column].mode()[0])
        
    # Convert hierarchical categorical variables to numerical using one-hot encoding
    map_level = {
      'Low': 0,
      'Medium': 1,
      'High': 2
    }
    
    hc_columns = ['Parental_Involvement', 'Motivation_Level']
    
    for column in hc_columns:
        df[f'{column}_encoded'] = df[column].map(map_level)
        
    # Convert categorical variables to numerical using one-hot encoding
    df = pd.get_dummies(df, columns=['School_Type', 'Internet_Access'], drop_first=False, dtype=int)
    
    # Clean atypical values in Exam_Score column
    over_100 = (df['Exam_Score'] > 100).sum()
    print(f"\nRegistros con calificación mayor a 100 en 'Exam_Score': {over_100}")
    
    under_100 = (df['Exam_Score'] < 0).sum()
    print(f"Registros con calificación menor a 0 en 'Exam_Score': {under_100}")
    
    df.loc[df['Exam_Score'] > 100, 'Exam_Score'] = 100
    df.loc[df['Exam_Score'] < 0, 'Exam_Score'] = 0
    
    print("\nValor mínimo y máximo de 'Exam_Score' después de la limpieza:")
    print(f"Valor mínimo: {df['Exam_Score'].min()}")
    print(f"Valor máximo: {df['Exam_Score'].max()}")
    
    # Standard scaling of 'Hours_Studied' and 'Attendance'
    cols_to_scale = ['Hours_Studied', 'Attendance']
    scaler = StandardScaler()
    
    for column in cols_to_scale:
        df[f'{column}_scaled'] = scaler.fit_transform(df[[column]])
    
    print("\nDescripción estadística del DataFrame después de la limpieza:")
    print(df['Exam_Score'].describe())
    return df

df = clean_data()

Valores faltantes en 'Teacher_Quality': 78
Valores faltantes en 'Parental_Education_Level': 90
Valores faltantes en 'Distance_from_Home': 67

Registros con calificación mayor a 100 en 'Exam_Score': 1
Registros con calificación menor a 0 en 'Exam_Score': 0

Valor mínimo y máximo de 'Exam_Score' después de la limpieza:
Valor mínimo: 55
Valor máximo: 100

Descripción estadística del DataFrame después de la limpieza:
count    6607.000000
mean       67.235508
std         3.889161
min        55.000000
25%        65.000000
50%        67.000000
75%        69.000000
max       100.000000
Name: Exam_Score, dtype: float64


/var/folders/n7/85qppsxd4pd_jb_gjtbyl0600000gn/T/ipykernel_30494/953104922.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_objects = df.select_dtypes(include=['object'])


In [25]:
print("Descripción estadística del DataFrame:")
df.describe(include="all")

Descripción estadística del DataFrame:


,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Tutoring_Sessions,Family_Income,...,Gender,Exam_Score,Parental_Involvement_encoded,Motivation_Level_encoded,School_Type_Private,School_Type_Public,Internet_Access_No,Internet_Access_Yes,Hours_Studied_scaled,Attendance_scaled
count,6607.000000,6607.000000,6607,6607,6607,6607.00000,6607.000000,6607,6607.000000,6607,...,6607,6607.000000,6607.000000,6607.000000,6607.000000,6607.000000,6607.000000,6607.000000,6.607000e+03,6.607000e+03
unique,NaN,NaN,3,3,2,NaN,NaN,3,NaN,3,...,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,Medium,Medium,Yes,NaN,NaN,Medium,NaN,Low,...,Male,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,3362,3319,3938,NaN,NaN,3351,NaN,2672,...,3814,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,19.975329,79.977448,NaN,NaN,NaN,7.02906,75.070531,NaN,1.493719,NaN,...,NaN,67.235508,1.086423,0.906463,0.304071,0.695929,0.075526,0.924474,-1.161474e-16,-3.089199e-16
std,5.990594,11.547475,NaN,NaN,NaN,1.46812,14.399784,NaN,1.230570,NaN,...,NaN,3.889161,0.695521,0.695798,0.460048,0.460048,0.264258,0.264258,1.000076e+00,1.000076e+00
min,1.000000,60.000000,NaN,NaN,NaN,4.00000,50.000000,NaN,0.000000,NaN,...,NaN,55.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-3.167760e+00,-1.730158e+00
25%,16.000000,70.000000,NaN,NaN,NaN,6.00000,63.000000,NaN,1.000000,NaN,...,NaN,65.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,-6.636453e-01,-8.641026e-01
50%,20.000000,80.000000,NaN,NaN,NaN,7.00000,75.000000,NaN,1.000000,NaN,...,NaN,67.000000,1.000000,1.000000,0.000000,1.000000,0.000000,1.000000,4.118568e-03,1.953115e-03
75%,24.000000,90.000000,NaN,NaN,NaN,8.00000,88.000000,NaN,2.000000,NaN,...,NaN,69.000000,2.000000,1.000000,1.000000,1.000000,0.000000,1.000000,6.718825e-01,8.680088e-01


In [26]:
print("Validación de cero valores nulos en el DataFrame:")
print(df.isnull().sum())

Validación de cero valores nulos en el DataFrame:
Hours_Studied                   0
Attendance                      0
Parental_Involvement            0
Access_to_Resources             0
Extracurricular_Activities      0
Sleep_Hours                     0
Previous_Scores                 0
Motivation_Level                0
Tutoring_Sessions               0
Family_Income                   0
Teacher_Quality                 0
Peer_Influence                  0
Physical_Activity               0
Learning_Disabilities           0
Parental_Education_Level        0
Distance_from_Home              0
Gender                          0
Exam_Score                      0
Parental_Involvement_encoded    0
Motivation_Level_encoded        0
School_Type_Private             0
School_Type_Public              0
Internet_Access_No              0
Internet_Access_Yes             0
Hours_Studied_scaled            0
Attendance_scaled               0
dtype: int64


# Conclusión y Justificación de Cambios

**Registros nuevos**
* **Parental_Involvement_Encoded:** *Nivel de apoyo académico por parte de los padres medido en valores númericos (del 0 al 2)*
* **Motivation_Level_Encoded:** *Niveo de motivación del estudiante medido en valores númericos (del 0 al 2)*
* **School_Type_Private:** *Si el estudiante atiende escuela privada (0 siendo Falso y 1 siendo Verdadero)*
* **School_Type_Public:** *Si el estudiante atiende escuela pública (0 siendo Falso y 1 siendo Verdadero)*
* **Internet_Access_No:** *Si el estudiante no posee acceso a internet (0 siendo Falso y 1 siendo Verdadero)*
* **Internet_Access_Yes:** *Si el estudiante sí posee acceso a internet (0 siendo Falso y 1 siendo Verdadero)*
* **Hours_Studied_Scaled:** *Número de horas de estudio por semana estandarizadas. Centrado en 0 con una desviación estándar de 1.*
* **Attendance_Scaled:** *Porcentaje de asistencia en el periodo estandarizado. Centrado en 0 con una desviación estándar de 1.*